# Notebook 2 — Anatomía de un Prompt
**Clase 2: Ingeniería de Prompts** | IA Generativa

## Objetivos
- Entender la estructura sistema/usuario/asistente
- Manejar `SystemMessage`, `HumanMessage` y `AIMessage` en LangChain
- Ver el efecto de cada componente del prompt de forma aislada
- Experimentar con prompts con y sin instrucción de sistema

---

In [ ]:
!pip install langchain langchain-google-genai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.7,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

print("✅ Modelo listo")

## 1. Los tres roles de un prompt

```
┌────────────────────────────────────────────────────────┐
│                    ANATOMÍA DE UN PROMPT                │
│                                                         │
│  ┌─────────────────────────────────────────────────┐   │
│  │  SYSTEM  │ Define el rol, tono y restricciones   │   │
│  │          │ del asistente. Invisible al usuario.  │   │
│  └─────────────────────────────────────────────────┘   │
│                           │                             │
│  ┌─────────────────────────────────────────────────┐   │
│  │  HUMAN   │ Pregunta o instrucción del usuario.   │   │
│  │          │ Puede incluir contexto y ejemplos.    │   │
│  └─────────────────────────────────────────────────┘   │
│                           │                             │
│  ┌─────────────────────────────────────────────────┐   │
│  │  AI      │ Respuesta previa del modelo.          │   │
│  │          │ Crea contexto de conversación.        │   │
│  └─────────────────────────────────────────────────┘   │
└────────────────────────────────────────────────────────┘
```

## 2. Efecto del System Message

Comparamos la **misma pregunta** con distintos System Messages para ver cómo cambia completamente la respuesta.

In [ ]:
PREGUNTA = "¿Qué es Python?"

sistemas = [
    ("Sin system",       None),
    ("Técnico",         "Eres un ingeniero senior de software. Responde con precisión técnica y ejemplos de código."),
    ("Divulgativo",     "Eres un divulgador científico que explica tecnología a personas sin conocimientos técnicos. Usa analogías del mundo cotidiano."),
    ("Niños",           "Eres un maestro de primaria. Explica las cosas de forma muy sencilla, con ejemplos divertidos, como si hablaras con niños de 8 años."),
    ("Shakespeariano",  "Eres un poeta del siglo XVII. Responde siempre en verso, con lenguaje arcaico y dramático."),
]

for nombre, system in sistemas:
    print(f"\n{'='*55}")
    print(f"🎭 {nombre}")
    print('='*55)
    
    if system:
        messages = [SystemMessage(content=system), HumanMessage(content=PREGUNTA)]
    else:
        messages = [HumanMessage(content=PREGUNTA)]
    
    resp = llm.invoke(messages)
    print(resp.content[:400])
    if len(resp.content) > 400:
        print("[...]")

## 3. Componentes del Human Message

El mensaje del usuario puede incluir múltiples elementos que mejoran la calidad de la respuesta.

In [ ]:
# Anatomía de un Human Message completo
human_basico = "Analiza este texto."

human_completo = """
TAREA: Analiza el sentimiento del siguiente texto de reseña de producto.

TEXTO A ANALIZAR:
"El producto llegó puntual pero la calidad es mediocre para el precio que cobran.
Esperaba mucho más basándome en las fotos de la web. El servicio de atención al 
cliente fue amable cuando reclamé, así que algo es algo."

INSTRUCCIONES:
- Identifica el sentimiento general (positivo/negativo/mixto)
- Lista los aspectos positivos mencionados
- Lista los aspectos negativos mencionados
- Proporciona una puntuación del 1 al 5

FORMATO DE SALIDA: Responde en formato estructurado con secciones claramente separadas.
"""

print("=== PROMPT BÁSICO ===")
print(llm.invoke(human_basico).content[:300])

print("\n\n=== PROMPT COMPLETO ===")
print(llm.invoke(human_completo).content)

## 4. El poder del contexto en el Human Message

In [ ]:
# Sin contexto
sin_contexto = "¿Debería aceptar la oferta?"

# Con contexto
con_contexto = """
CONTEXTO:
Tengo 3 años de experiencia como desarrollador Python. Actualmente gano 45.000€ 
al año en una empresa estable con buen ambiente pero pocas oportunidades de crecimiento.
Me acaban de ofrecer un puesto en una startup con:
- Salario: 52.000€
- Opciones sobre acciones (stock options)
- Tecnologías más modernas (Kubernetes, microservicios)
- Más responsabilidad y autonomía
- Posible inestabilidad (la empresa lleva 2 años y aún no es rentable)

PREGUNTA: ¿Debería aceptar la oferta?

Por favor, presenta los pros y contras de forma objetiva sin decirme directamente qué hacer.
"""

print("=== SIN CONTEXTO ===")
print(llm.invoke(sin_contexto).content)

print("\n\n=== CON CONTEXTO ===")
print(llm.invoke(con_contexto).content)

## 5. El AI Message para guiar la respuesta

Podemos usar `AIMessage` para "pre-rellenar" una respuesta y así controlar el formato o dirección.

In [ ]:
# Técnica: forzar respuesta en JSON usando AIMessage como prefijo
messages_con_prefijo = [
    SystemMessage(content="Eres un extractor de información. Siempre respondes en JSON válido."),
    HumanMessage(content="Extrae la información de este texto: 'Ana García, 32 años, vive en Madrid, trabaja como ingeniera de software en Telefónica desde 2019.'"),
    AIMessage(content='{"nombre":')  # Prefijo que fuerza el formato
]

messages_sin_prefijo = [
    SystemMessage(content="Eres un extractor de información. Siempre respondes en JSON válido."),
    HumanMessage(content="Extrae la información de este texto: 'Ana García, 32 años, vive en Madrid, trabaja como ingeniera de software en Telefónica desde 2019.'")
]

print("=== CON PREFIJO JSON ===")
resp = llm.invoke(messages_con_prefijo)
print('{"nombre":' + resp.content)  # Reconstruimos el JSON completo

print("\n=== SIN PREFIJO ===")
print(llm.invoke(messages_sin_prefijo).content)

## 6. Historial de conversación real

In [ ]:
# Simular una conversación multi-turno
historial = [
    SystemMessage(content="Eres un tutor de programación paciente y motivador. "
                          "Recuerda lo que el estudiante te cuenta sobre sí mismo.")
]

def chat(mensaje_usuario: str) -> str:
    """Mantiene el historial y devuelve la respuesta del modelo."""
    historial.append(HumanMessage(content=mensaje_usuario))
    respuesta = llm.invoke(historial)
    historial.append(AIMessage(content=respuesta.content))
    return respuesta.content

# Conversación
turnos = [
    "Hola, me llamo Luis y soy principiante en Python. Llevo 2 semanas aprendiendo.",
    "Tengo problemas para entender las listas. ¿Puedes explicarme cómo funcionan?",
    "¿Y para qué usaría una lista en un programa real?",
    "¿Recuerdas cuánto llevo aprendiendo Python?",  # Test de memoria
]

for turno in turnos:
    print(f"\n👤 {turno}")
    print(f"🤖 {chat(turno)[:400]}")
    print()

## 7. Inspeccionando el historial

In [ ]:
print(f"Mensajes en el historial: {len(historial)}\n")

for i, msg in enumerate(historial):
    tipo = msg.__class__.__name__
    icono = {"SystemMessage": "⚙️", "HumanMessage": "👤", "AIMessage": "🤖"}.get(tipo, "❓")
    print(f"[{i}] {icono} {tipo}")
    print(f"    {msg.content[:100]}{'...' if len(msg.content) > 100 else ''}")
    print()

# Calcular tokens aproximados
total_chars = sum(len(m.content) for m in historial)
tokens_aprox = total_chars // 4  # ~4 chars por token
print(f"📊 Caracteres totales: {total_chars}")
print(f"📊 Tokens aproximados: {tokens_aprox}")
print("⚠️  El historial crece con cada turno — ¡hay que gestionarlo!")

## 8. El problema del contexto infinito y soluciones

In [ ]:
# Estrategia 1: Ventana deslizante (mantener solo los últimos N mensajes)
def chat_con_ventana(mensaje: str, historial_completo: list, ventana: int = 6) -> str:
    """Mantiene solo los últimos `ventana` mensajes (excluyendo system)."""
    system = [m for m in historial_completo if isinstance(m, SystemMessage)]
    resto = [m for m in historial_completo if not isinstance(m, SystemMessage)]
    
    # Mantener solo los últimos N mensajes de la conversación
    historial_truncado = system + resto[-ventana:]
    historial_truncado.append(HumanMessage(content=mensaje))
    
    respuesta = llm.invoke(historial_truncado)
    historial_completo.append(HumanMessage(content=mensaje))
    historial_completo.append(AIMessage(content=respuesta.content))
    return respuesta.content

print("Estrategia de ventana deslizante:")
print(f"  Historial completo tiene {len(historial)} mensajes")
print(f"  Solo enviamos los últimos 6 al modelo")
print("  → Coste constante independientemente de cuántos turnos haya")

## 9. Análisis comparativo: impacto de cada componente

In [ ]:
TAREA = "Resume las ventajas de usar Python para ciencia de datos en 3 puntos."

variantes = [
    {
        "nombre": "Solo instrucción",
        "messages": [HumanMessage(content=TAREA)]
    },
    {
        "nombre": "Con sistema experto",
        "messages": [
            SystemMessage(content="Eres un científico de datos con 10 años de experiencia en producción."),
            HumanMessage(content=TAREA)
        ]
    },
    {
        "nombre": "Con sistema + formato",
        "messages": [
            SystemMessage(content="Eres un científico de datos con 10 años de experiencia en producción. "
                                  "Responde siempre con bullet points concisos y añade un ejemplo práctico por punto."),
            HumanMessage(content=TAREA)
        ]
    },
    {
        "nombre": "Con sistema + formato + audiencia",
        "messages": [
            SystemMessage(content="Eres un científico de datos con 10 años de experiencia en producción. "
                                  "Responde siempre con bullet points concisos y añade un ejemplo práctico por punto. "
                                  "Tu audiencia son directivos de empresa sin background técnico."),
            HumanMessage(content=TAREA)
        ]
    },
]

for v in variantes:
    print(f"\n{'='*60}")
    print(f"📝 {v['nombre']}")
    print('='*60)
    print(llm.invoke(v['messages']).content)

## Resumen

| Componente | Función | Impacto |
|---|---|---|
| `SystemMessage` | Define rol, tono y restricciones | 🔴 Alto — cambia toda la personalidad |
| `HumanMessage` | Pregunta/instrucción + contexto | 🟡 Medio — determina la tarea |
| `AIMessage` | Historial o prefijo de respuesta | 🟡 Medio — guía el formato/dirección |
| Contexto en Human | Información adicional relevante | 🟡 Medio — mejora la precisión |
| Audiencia en System | A quién va dirigida la respuesta | 🔴 Alto — cambia vocabulario y profundidad |

**➡️ Siguiente:** Notebook 3 — PromptTemplates y reutilización

### 🏋️ Ejercicios
1. Escribe un `SystemMessage` que haga que el modelo responda siempre en formato de tabla markdown
2. ¿Puedes hacer que el modelo "recuerde" que eres de Barcelona usando solo el primer `HumanMessage`?
3. Experimenta con `AIMessage` de prefijo para forzar que la respuesta empiece con "En resumen:"